In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, BooleanType

schema = StructType([
    StructField("customer_id",           IntegerType(), True),
    StructField("customer_name",         StringType(),  True),
    StructField("city",                  StringType(),  True),
    StructField("signup_date",           DateType(),    True),
    StructField("last_order_date",       DateType(),    True),
    StructField("total_orders",          IntegerType(), True),
    StructField("avg_order_value",       DoubleType(),  True),
    StructField("total_spent",           DoubleType(),  True),
    StructField("preferred_cuisine",     StringType(),  True),
    StructField("payment_method",        StringType(),  True),
    StructField("is_premium_member",     BooleanType(), True),
    StructField("review_text",           StringType(),  True),
    StructField("rating",                DoubleType(),  True),
    StructField("days_since_last_order", IntegerType(), True),
])

source_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/Volumes/demo_catalog/delta_demo/delta_volume/food_delivery_customers.txt")

print(f"Total customers loaded: {source_df.count()}")
source_df.show(5)

In [0]:
##spark.sql("CREATE CATALOG IF NOT EXISTS pipeline_catalog")
##spark.sql("CREATE SCHEMA IF NOT EXISTS pipeline_catalog.tabular_pipeline")

In [0]:
%python
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_catalog.tabular_pipeline.customer_summary")

print("customer_summary Delta table created successfully")

# 1. Engineering numerical features

In [0]:
%python
from pyspark.sql.functions import col, when

numerical_features_df = source_df \
    .withColumn(
        "order_frequency_segment",
        when(col("total_orders") < 30, "Low")
        .when(col("total_orders") < 80, "Medium")
        .otherwise("High")
    ) \
    .withColumn(
        "rating_segment",
        when(col("rating") < 3.0, "Low")
        .when(col("rating") < 4.0, "Medium")
        .otherwise("High")
    ) \
    .withColumn(
        "high_value_customer",
        when(col("total_spent") > 50000, 1).otherwise(0)
    )

numerical_features_df.select(
    "customer_id", "total_orders", "order_frequency_segment",
    "rating", "rating_segment", "total_spent", "high_value_customer"
).show(10)

In [0]:
%python
# One-hot encoding for payment_method (small, manageable number of categories)
payment_methods = [row["payment_method"] for row in numerical_features_df.select("payment_method").distinct().collect()]
print(f"Distinct payment methods: {payment_methods}")

encoded_df = numerical_features_df
for method in payment_methods:
    col_name = "payment_" + method.lower().replace(" ", "_")
    encoded_df = encoded_df.withColumn(col_name, when(col("payment_method") == method, 1).otherwise(0))

# Frequency encoding for preferred_cuisine (higher cardinality, one-hot would be too wide)
cuisine_freq_df = encoded_df.groupBy("preferred_cuisine").count() \
    .withColumnRenamed("count", "cuisine_frequency")
encoded_df = encoded_df.join(cuisine_freq_df, on="preferred_cuisine", how="left")

# Converting the boolean is_premium_member into a numeric 0/1 column
encoded_df = encoded_df.withColumn("is_premium_flag", col("is_premium_member").cast("int"))

encoded_df.select(
    "customer_id", "payment_method", "payment_upi", "payment_credit_card",
    "preferred_cuisine", "cuisine_frequency", "is_premium_member", "is_premium_flag"
).show(10)

## Writing the Feature-Engineered Table

In [0]:
%python
encoded_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_catalog.tabular_pipeline.customer_features_v1")

print("customer_features_v1 Delta table created successfully")

# 2. Building Recency and Tenure Features

## 2.1 Reading the Source Customer Data

In [0]:
%python
source_df = spark.table("pipeline_catalog.tabular_pipeline.customer_summary")
source_df.select("customer_id", "signup_date", "last_order_date", "days_since_last_order").show(10)

###2.2Tenure is a single number summarizing how long a customer has been engaging with platform, rather than two separatee dates.

In [0]:
%python
from pyspark.sql.functions import datediff, current_date, col

tenure_df = source_df.withColumn(
    "tenure_days",
    datediff(col("last_order_date"), col("signup_date"))
)

tenure_df.select("customer_id", "signup_date", "last_order_date", "tenure_days").show(10)

###2.3Recency segment more easy for model than a raw day count

In [0]:
%python
from pyspark.sql.functions import when

recency_df = tenure_df.withColumn(
    "recency_segment",
    when(col("days_since_last_order") < 30, "Active")
    .when(col("days_since_last_order") < 90, "At Risk")
    .otherwise("Dormant")
)

recency_df.select("customer_id", "days_since_last_order", "recency_segment").show(10)

##2.4 Extracting Signup Month and Quarterto help model pick up on seasonal paterns

In [0]:
%python
from pyspark.sql.functions import month, quarter

calendar_df = recency_df \
    .withColumn("signup_month", month(col("signup_date"))) \
    .withColumn("signup_quarter", quarter(col("signup_date")))

calendar_df.select("customer_id", "signup_date", "signup_month", "signup_quarter").show(10)

## 2.5 Calculating an Engagement Ratio by dividing days since last order by tenure days

In [0]:
%python
from pyspark.sql.functions import round as _round

engagement_df = calendar_df.withColumn(
    "engagement_ratio",
    _round(col("days_since_last_order") / col("tenure_days"), 3)
)

engagement_df.select(
    "customer_id", "tenure_days", "days_since_last_order", "engagement_ratio"
).show(10)

Customer 1008 is someone who is still ordering recentlly, with long history
Customer 1009 barrely engaged before disappearing

## 2.6 Writing the Temporal Features Table

In [0]:
%python
engagement_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_catalog.tabular_pipeline.customer_temporal_features")

print("customer_temporal_features Delta table created successfully")


# 3. Handling Skewness, Scaling, and Encoding for ML

## 3.1 Combining Numerical, Categorical, and Temporal Features

In [0]:
%python
features_df = spark.table("pipeline_catalog.tabular_pipeline.customer_features_v1")
temporal_df = spark.table("pipeline_catalog.tabular_pipeline.customer_temporal_features")

combined_df = features_df.join(
    temporal_df.select("customer_id", "tenure_days", "recency_segment",
                        "signup_month", "signup_quarter", "engagement_ratio"),
    on="customer_id",
    how="inner"
)

print(f"Combined records: {combined_df.count()}")
combined_df.select("customer_id", "total_spent", "total_orders", "avg_order_value",
                    "tenure_days", "engagement_ratio").show(5)

## 3.2 Checking Skewness Before Doing Anything
Skewness is simply a measure of how much a dataset is pulled toward one side because ​of these unusually high or low values. 
- Close to zero, the data is fairly balanced.
-  ​A positive skewness means there are a few much larger values creating a tail on the ​right, - A negative skewness means there are a few much smaller values creating a tail ​on the left.


In [0]:
%python
from pyspark.sql.functions import skewness

combined_df.select(
    skewness("total_spent").alias("total_spent_skew"),
    skewness("total_orders").alias("total_orders_skew"),
    skewness("avg_order_value").alias("avg_order_value_skew"),
    skewness("engagement_ratio").alias("engagement_ratio_skew")
).show()

 ​We can see that total orders and average order value both have skewness. ​Value is very close to zero, so their distributions are quite balanced. ​Total spent has a skewness of 0.862, indicating a slight right skew, but it's still not a ​major concern. the total spent had a value of 0.862. ​That tells us there are a few customers who have spent more than others, but the difference ​isn't large enough to significantly distort the dat
  ​However, the engagement ratio has a skewness of 8.557, which is extremely high. ​This tells us that while most customers have relatively similar engagement ratios, a small ​number of customers have exceptionally large values that are stretching the distribution ​significantly. 

## Applying Log Transformation 

In [0]:
%python
from pyspark.sql.functions import log1p, col, round as _round

log_transformed_df = combined_df \
    .withColumn("engagement_ratio_log", _round(log1p(col("engagement_ratio")), 3))

log_transformed_df.select(
    "customer_id", "engagement_ratio", "engagement_ratio_log"
).show(10)

## Verifying Skewness After the Transformation

In [0]:
%python
log_transformed_df.select(
    skewness("engagement_ratio_log").alias("engagement_ratio_log_skew")
).show()

## 3.3. Applying Min-Max Scaling

In [0]:
%python
from pyspark.sql.functions import min as _min, max as _max, round as _round

stats = log_transformed_df.select(
    _min("total_spent").alias("min_spent"), _max("total_spent").alias("max_spent"),
    _min("total_orders").alias("min_orders"), _max("total_orders").alias("max_orders"),
    _min("avg_order_value").alias("min_aov"), _max("avg_order_value").alias("max_aov"),
    _min("engagement_ratio_log").alias("min_engage"), _max("engagement_ratio_log").alias("max_engage"),
).collect()[0]

scaled_df = log_transformed_df \
    .withColumn("total_spent_scaled",
        _round((col("total_spent") - stats["min_spent"]) / (stats["max_spent"] - stats["min_spent"]), 4)) \
    .withColumn("total_orders_scaled",
        _round((col("total_orders") - stats["min_orders"]) / (stats["max_orders"] - stats["min_orders"]), 4)) \
    .withColumn("avg_order_value_scaled",
        _round((col("avg_order_value") - stats["min_aov"]) / (stats["max_aov"] - stats["min_aov"]), 4)) \
    .withColumn("engagement_ratio_scaled",
        _round((col("engagement_ratio_log") - stats["min_engage"]) / (stats["max_engage"] - stats["min_engage"]), 4))

scaled_df.select(
    "customer_id", "total_spent_scaled", "total_orders_scaled",
    "avg_order_value_scaled", "engagement_ratio_scaled"
).show(10)

##3.4 Applying Ordinal Encoding to Ordered Categories

In [0]:
%python
from pyspark.sql.functions import when, col

ordinal_df = scaled_df \
    .withColumn("order_frequency_ordinal",
        when(col("order_frequency_segment") == "Low", 0)
        .when(col("order_frequency_segment") == "Medium", 1)
        .otherwise(2)
    ) \
    .withColumn("rating_ordinal",
        when(col("rating_segment") == "Low", 0)
        .when(col("rating_segment") == "Medium", 1)
        .otherwise(2)
    ) \
    .withColumn("recency_ordinal",
        when(col("recency_segment") == "Active", 0)
        .when(col("recency_segment") == "At Risk", 1)
        .otherwise(2)
    )

ordinal_df.select(
    "customer_id", "order_frequency_segment", "order_frequency_ordinal",
    "rating_segment", "rating_ordinal", "recency_segment", "recency_ordinal"
).show(10)

##3.5 Writing the Final Feature Table

In [0]:
%python
ordinal_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_catalog.tabular_pipeline.customer_features_final")

print("customer_features_final Delta table created successfully")

# 4. Creating and Writing to a Feature Table in Databricks

## Reviewing the Table Before Registration

In [0]:
%sql
SELECT * FROM pipeline_catalog.tabular_pipeline.customer_features_final LIMIT 5;

## Confirming No Primary Key Exists Yet

In [0]:
%sql
DESCRIBE TABLE EXTENDED pipeline_catalog.tabular_pipeline.customer_features_final;

primary key cannot be declared on a column allowing NULLS

## Setting customer_id to NOT NULL

In [0]:
%sql
ALTER TABLE pipeline_catalog.tabular_pipeline.customer_features_final
ALTER COLUMN customer_id SET NOT NULL;

## Declaring the Primary Key and transform it to feature table

In [0]:
%sql
ALTER TABLE pipeline_catalog.tabular_pipeline.customer_features_final
ADD CONSTRAINT customer_features_final_pk PRIMARY KEY (customer_id);

%md
## Adding a Descriptive Comment for Discoverability

In [0]:
%sql
COMMENT ON TABLE pipeline_catalog.tabular_pipeline.customer_features_final IS
'ML-ready customer feature table with engineered numerical, categorical, and temporal features. Primary key: customer_id.';

# Churn feature engineering with PySpark